# Exploratory Analysis & Classic Baseline

1. **Classic pairs trading baseline** — top-150 liquid, 100% coverage, top-300 correlated pairs (ρ>0.5), Breusch-Pagan filter, walk-forward backtest
2. **Linear autoregression** — per-stock AR(15) to determine optimal lookback for future RNN models

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
RAW_DIR = os.path.join(PROJECT_ROOT, 'data')  # bars_30m_YYYY.parquet live here

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DIR}")
print(f"Files: {sorted(os.listdir(RAW_DIR))}")

## Data loading utilities

Build price and return panels directly from the raw parquet files. No dependency on Daniel's module (his paths are broken).

In [ ]:
# ---- Data loading (self-contained, no daniel/ dependency) ----

OPEN_BAR = "09:30"
BARS_PER_DAY = 13
INTRADAY_BARS_PER_DAY = 12
BARS_PER_YEAR = INTRADAY_BARS_PER_DAY * 252  # 3024

def load_bars(years, regular_hours_only=True, columns=None):
    """Load raw bar parquets for given years."""
    if isinstance(years, int):
        years = [years]
    frames = []
    for y in years:
        path = os.path.join(RAW_DIR, f"bars_30m_{y}.parquet")
        read_cols = columns
        drop_flag = False
        if read_cols is not None:
            read_cols = list(dict.fromkeys(read_cols))
            if regular_hours_only and "is_regular_hours" not in read_cols:
                read_cols = read_cols + ["is_regular_hours"]
                drop_flag = True
        df = pd.read_parquet(path, columns=read_cols)
        if regular_hours_only:
            df = df.loc[df["is_regular_hours"]]
        if drop_flag:
            df = df.drop(columns=["is_regular_hours"])
        frames.append(df)
    out = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
    return out.reset_index(drop=True)


def build_price_panel(years, price_col="close"):
    """Wide price panel: rows=timestamps, cols=symbols, values=close."""
    if isinstance(years, int):
        years = [years]
    cols = ["symbol", "timestamp_utc", price_col]
    per_year = []
    for y in years:
        df = load_bars(y, regular_hours_only=True, columns=cols)
        df = df.drop_duplicates(["timestamp_utc", "symbol"], keep="last")
        wide = df.pivot(index="timestamp_utc", columns="symbol", values=price_col)
        per_year.append(wide)
    panel = pd.concat(per_year, axis=0) if len(per_year) > 1 else per_year[0]
    panel = panel.sort_index().reindex(sorted(panel.columns), axis=1)
    panel.index.name = "timestamp_utc"
    panel.columns.name = "symbol"
    return panel


def build_return_panel(price_panel, drop_overnight=True):
    """Bar-to-bar log returns. Overnight (09:30) rows set to NaN."""
    ret = np.log(price_panel / price_panel.shift(1))
    if drop_overnight:
        ny_time = price_panel.index.tz_convert("America/New_York").strftime("%H:%M")
        overnight_mask = np.asarray(ny_time) == OPEN_BAR
        ret.loc[overnight_mask, :] = np.nan
    return ret


def liquid_universe(years, top_n=150, date_range=None):
    """Top-N symbols by median per-bar dollar volume."""
    if isinstance(years, int):
        years = [years]
    cols = ["symbol", "date_ny", "close", "volume"]
    parts = []
    for y in years:
        df = load_bars(y, regular_hours_only=True, columns=cols)
        if date_range is not None:
            start, end = date_range
            df = df.loc[(df["date_ny"] >= start) & (df["date_ny"] <= end)]
        if df.empty:
            continue
        dv = (df["close"] * df["volume"]).to_numpy()
        parts.append(pd.DataFrame({"symbol": df["symbol"].to_numpy(), "dollar_vol": dv}))
    if not parts:
        return []
    allbars = pd.concat(parts, ignore_index=True)
    med = allbars.groupby("symbol")["dollar_vol"].median().sort_values(ascending=False)
    return med.head(top_n).index.tolist()


# ---- Signal utilities (inlined from daniel/src/signal.py) ----

from dataclasses import dataclass

@dataclass
class PairParams:
    sym_a: str
    sym_b: str
    beta: float
    alpha: float
    mu: float
    sigma: float
    half_life: float
    extra: dict = None


def hedge_ratio_ols(logp_a, logp_b):
    df = pd.concat([logp_a, logp_b], axis=1, keys=["a", "b"]).dropna()
    if len(df) < 10:
        return np.nan, np.nan
    X = sm.add_constant(df["b"].to_numpy())
    res = sm.OLS(df["a"].to_numpy(), X).fit()
    return float(res.params[0]), float(res.params[1])


def build_spread(logp_a, logp_b, alpha, beta):
    return logp_a - (alpha + beta * logp_b)


def half_life_ou(spread):
    s = pd.Series(spread).dropna()
    if len(s) < 20:
        return np.nan
    s_lag = s.shift(1)
    ds = (s - s_lag).dropna()
    s_lag = s_lag.loc[ds.index]
    X = sm.add_constant(s_lag.to_numpy())
    try:
        res = sm.OLS(ds.to_numpy(), X).fit()
    except Exception:
        return np.nan
    b = float(res.params[1])
    if b >= 0 or (1.0 + b) <= 0:
        return np.nan
    hl = -np.log(2.0) / np.log(1.0 + b)
    return float(hl) if np.isfinite(hl) and hl > 0 else np.nan


def fit_pair_params(train_close, sym_a, sym_b):
    if sym_a not in train_close.columns or sym_b not in train_close.columns:
        return None
    logp_a = np.log(train_close[sym_a])
    logp_b = np.log(train_close[sym_b])
    alpha, beta = hedge_ratio_ols(logp_a, logp_b)
    if not np.isfinite(alpha) or not np.isfinite(beta) or beta == 0:
        return None
    spread = build_spread(logp_a, logp_b, alpha, beta).dropna()
    if len(spread) < 20:
        return None
    mu = float(spread.mean())
    sigma = float(spread.std(ddof=1))
    if not np.isfinite(sigma) or sigma <= 0:
        return None
    hl = half_life_ou(spread)
    return PairParams(sym_a, sym_b, beta, alpha, mu, sigma, hl)


def zscore_frozen(spread, mu, sigma):
    return (spread - mu) / sigma


def zscore_rolling(spread, window, min_periods=None):
    window = max(int(window), 2)
    if min_periods is None:
        min_periods = window
    m = spread.rolling(window, min_periods=min_periods).mean()
    sd = spread.rolling(window, min_periods=min_periods).std(ddof=1)
    return (spread - m) / sd


def generate_positions(z, z_entry=2.0, z_exit=0.5, z_stop=4.0):
    z = pd.Series(z)
    pos = np.zeros(len(z), dtype=float)
    state = 0.0
    zv = z.to_numpy()
    for i in range(len(zv)):
        zi = zv[i]
        if not np.isfinite(zi):
            state = 0.0
        elif state == 0.0:
            if zi >= z_entry:
                state = -1.0
            elif zi <= -z_entry:
                state = 1.0
        else:
            if abs(zi) <= z_exit or abs(zi) >= z_stop:
                state = 0.0
        pos[i] = state
    return pd.Series(pos, index=z.index)


def pair_returns(test_close, test_logret, params, z_entry=2.0, z_exit=0.5,
                 z_stop=4.0, cost_per_side=0.0005, use_rolling_z=False):
    a, b, beta = params.sym_a, params.sym_b, params.beta
    if a not in test_close.columns or b not in test_close.columns:
        idx = test_close.index
        zero = pd.Series(0.0, index=idx)
        return {"ret": zero, "gross_ret": zero, "turnover": zero,
                "position": zero, "trades": []}
    logp_a = np.log(test_close[a])
    logp_b = np.log(test_close[b])
    spread = build_spread(logp_a, logp_b, params.alpha, beta)
    if use_rolling_z and np.isfinite(params.half_life):
        win = int(max(round(params.half_life * 2), 5))
        z = zscore_rolling(spread, win)
    else:
        z = zscore_frozen(spread, params.mu, params.sigma)
    pos = generate_positions(z, z_entry, z_exit, z_stop)
    denom = 1.0 + abs(beta)
    w_a_unit = 1.0 / denom
    w_b_unit = -beta / denom
    pos_in = pos.shift(1).fillna(0.0)
    w_a = pos_in * w_a_unit
    w_b = pos_in * w_b_unit
    r_a = test_logret[a].reindex(spread.index, fill_value=0.0) if a in test_logret.columns else pd.Series(0.0, index=spread.index)
    r_b = test_logret[b].reindex(spread.index, fill_value=0.0) if b in test_logret.columns else pd.Series(0.0, index=spread.index)
    r_a = r_a.fillna(0.0)
    r_b = r_b.fillna(0.0)
    gross_ret = w_a * r_a + w_b * r_b
    dw_a = w_a.diff().abs().fillna(w_a.abs())
    dw_b = w_b.diff().abs().fillna(w_b.abs())
    turnover = dw_a + dw_b
    cost = turnover * cost_per_side
    net_ret = gross_ret - cost
    trades = _extract_trades(pos, gross_ret, cost, a, b)
    return {"ret": net_ret, "gross_ret": gross_ret, "turnover": turnover,
            "position": pos, "trades": trades}


def _extract_trades(pos, gross_ret, cost, sym_a, sym_b):
    trades = []
    pv = pos.to_numpy()
    idx = pos.index
    gr = gross_ret.reindex(idx).fillna(0.0).to_numpy()
    cs = cost.reindex(idx).fillna(0.0).to_numpy()
    in_trade = False
    entry_i = None
    side = 0.0
    for i in range(len(pv)):
        p = pv[i]
        if not in_trade and p != 0.0:
            in_trade = True; entry_i = i; side = p
        elif in_trade and (p == 0.0 or p != side):
            lo = entry_i + 1; hi = i + 1
            pnl = float(gr[lo:hi].sum() - cs[entry_i:hi].sum())
            trades.append({"sym_a": sym_a, "sym_b": sym_b, "side": side,
                           "entry_idx": idx[entry_i], "exit_idx": idx[i],
                           "holding_bars": i - entry_i, "pnl": pnl})
            if p != 0.0:
                in_trade = True; entry_i = i; side = p
            else:
                in_trade = False; side = 0.0
    if in_trade and entry_i is not None:
        i = len(pv) - 1; lo = entry_i + 1; hi = i + 1
        pnl = float(gr[lo:hi].sum() - cs[entry_i:hi].sum())
        trades.append({"sym_a": sym_a, "sym_b": sym_b, "side": side,
                       "entry_idx": idx[entry_i], "exit_idx": idx[i],
                       "holding_bars": i - entry_i, "pnl": pnl})
    return trades


# ---- Metrics (inlined) ----

def annualized_sharpe(returns):
    r = pd.Series(returns).dropna()
    if len(r) < 2:
        return 0.0
    sd = r.std(ddof=1)
    return float(r.mean() / sd * np.sqrt(BARS_PER_YEAR)) if sd > 0 else 0.0

def equity_curve(returns):
    return (1.0 + pd.Series(returns).fillna(0.0)).cumprod()

def total_return(returns):
    eq = equity_curve(returns)
    return float(eq.iloc[-1] - 1.0) if not eq.empty else 0.0

def max_drawdown(returns):
    eq = equity_curve(returns)
    if eq.empty:
        return 0.0
    return float((eq / eq.cummax() - 1.0).min())

def summarize(returns, turnover=None, trade_log=None, n_active_pairs=None):
    out = {
        "sharpe_annualized": annualized_sharpe(returns),
        "return_total": total_return(returns),
        "max_drawdown": max_drawdown(returns),
        "n_bars": len(returns),
    }
    if turnover is not None:
        t = pd.Series(turnover).fillna(0.0)
        out["turnover_annualized"] = float(t.mean() * BARS_PER_YEAR)
    if trade_log is not None and len(trade_log) > 0:
        out["n_trades"] = len(trade_log)
        out["trade_win_rate"] = float((trade_log["pnl"] > 0).mean())
        out["avg_holding_bars"] = float(trade_log["holding_bars"].mean())
    if n_active_pairs is not None:
        out["n_active_pairs"] = float(n_active_pairs)
    return out

print("Utilities loaded.")

## Build price & return panels from raw data (2016-2025)

In [ ]:
%%time
years = list(range(2016, 2026))
close_panel = build_price_panel(years)
logret_panel = build_return_panel(close_panel)
print(f"Close panel: {close_panel.shape}")
print(f"Log-return panel: {logret_panel.shape}")
print(f"Date range: {close_panel.index.min()} to {close_panel.index.max()}")

---
# Part 1: Classic Pairs Trading Baseline

Per fold:
1. Top-150 liquid symbols (median dollar volume on train window)
2. **100% coverage** filter (no missing bars in train window)
3. Pairwise correlation of log-returns → top 300 pairs with ρ > 0.5
4. OLS hedge ratio + Breusch-Pagan test → flag & drop heteroskedastic pairs
5. Half-life / spread-vol filters → trade surviving pairs on test window

In [ ]:
def get_liquid_universe_for_window(train_start, train_end, top_n=150):
    """Top-N symbols by median dollar volume on the train window."""
    y0, y1 = train_start.year, train_end.year
    years = list(range(y0, y1 + 1))
    date_range = (train_start.strftime('%Y-%m-%d'), train_end.strftime('%Y-%m-%d'))
    return liquid_universe(years, top_n=top_n, date_range=date_range)


def filter_100pct_coverage(panel_slice, symbols):
    """Keep only symbols with zero NaN bars in this window."""
    sub = panel_slice.reindex(columns=[s for s in symbols if s in panel_slice.columns])
    coverage = sub.notna().mean()
    keep = coverage[coverage >= 1.0].index.tolist()
    return keep


def top_correlated_pairs(train_logret, symbols, top_k=300, min_corr=0.5):
    """Top-k most correlated pairs from train log-returns."""
    R = train_logret[symbols].dropna(how='all', axis=1)
    corr = R.corr(min_periods=50)
    cols = list(corr.columns)
    cm = corr.to_numpy()
    n = len(cols)
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            c = cm[i, j]
            if np.isfinite(c) and c >= min_corr:
                pairs.append((cols[i], cols[j], float(c)))
    pairs.sort(key=lambda x: x[2], reverse=True)
    return pairs[:top_k], corr


def breusch_pagan_test(train_close, sym_a, sym_b):
    """OLS of log-prices + Breusch-Pagan test on residuals."""
    logp_a = np.log(train_close[sym_a]).dropna()
    logp_b = np.log(train_close[sym_b]).dropna()
    common = logp_a.index.intersection(logp_b.index)
    if len(common) < 60:
        return None
    y = logp_a.loc[common].to_numpy()
    X = sm.add_constant(logp_b.loc[common].to_numpy())
    try:
        ols_result = sm.OLS(y, X).fit()
        bp_stat, bp_pval, _, _ = het_breuschpagan(ols_result.resid, X)
        return bp_stat, bp_pval, ols_result
    except Exception:
        return None

In [ ]:
%%time
# Run the pair selection pipeline across all folds
fold_results = []
all_corr_values = []  # for histogram across all folds
fold_liquid_cache = {}  # cache liquid universes for Part 2

for fi, fold in enumerate(folds):
    tr_close = slice_window(close_panel, fold['train_start'], fold['train_end'])
    tr_logret = slice_window(logret_panel, fold['train_start'], fold['train_end'])

    # 1. Liquid universe
    liquid = get_liquid_universe_for_window(fold['train_start'], fold['train_end'])
    liquid = [s for s in liquid if s in tr_close.columns]
    fold_liquid_cache[fi] = liquid

    # 2. 100% coverage filter
    clean = filter_100pct_coverage(tr_close, liquid)

    # 3. Top correlated pairs
    tr_logret_clean = tr_logret.reindex(columns=clean)
    pairs, corr_mat = top_correlated_pairs(tr_logret_clean, clean, top_k=300, min_corr=0.5)

    # Store all pairwise correlations for histogram
    cm = corr_mat.to_numpy()
    iu = np.triu_indices(len(corr_mat), 1)
    all_corr_values.extend(cm[iu][np.isfinite(cm[iu])].tolist())

    # 4. Breusch-Pagan test on each pair
    tr_close_clean = tr_close.reindex(columns=clean)
    pair_info = []
    for a, b, rho in pairs:
        bp = breusch_pagan_test(tr_close_clean, a, b)
        if bp is None:
            continue
        bp_stat, bp_pval, ols_res = bp
        bp_pass = bp_pval > 0.05

        params = fit_pair_params(tr_close_clean, a, b)

        pair_info.append({
            'sym_a': a, 'sym_b': b, 'corr': rho,
            'bp_stat': bp_stat, 'bp_pval': bp_pval, 'bp_pass': bp_pass,
            'has_params': params is not None,
            'half_life': params.half_life if params else np.nan,
            'beta': params.beta if params else np.nan,
            'sigma': params.sigma if params else np.nan,
            'params': params,
        })

    fold_results.append({
        'fold_idx': fi,
        'train_start': fold['train_start'],
        'train_end': fold['train_end'],
        'test_start': fold['test_start'],
        'test_end': fold['test_end'],
        'n_liquid': len(liquid),
        'n_clean': len(clean),
        'n_candidate_pairs': len(pairs),
        'pair_info': pair_info,
    })

    n_bp_pass = sum(1 for p in pair_info if p['bp_pass'])
    print(f"Fold {fi:2d}: {len(liquid)} liquid -> {len(clean)} clean -> "
          f"{len(pairs)} pairs (ρ>0.5) -> {n_bp_pass} pass BP")

print(f"\nDone: {len(fold_results)} folds processed")

### Visualizations: Pair Selection

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram of ALL pairwise correlations across folds (top 150 liquid)
ax = axes[0, 0]
ax.hist(all_corr_values, bins=80, color='steelblue', alpha=0.8, edgecolor='none')
ax.axvline(0.5, color='red', ls='--', lw=1.5, label='ρ=0.5 cutoff')
ax.set_xlabel('Pairwise log-return correlation')
ax.set_ylabel('Count')
ax.set_title('Distribution of pairwise correlations (top-150 liquid, all folds)')
ax.legend()

# 2. Number of pairs passing each filter per fold
ax = axes[0, 1]
fold_ids = [f['fold_idx'] for f in fold_results]
n_candidates = [f['n_candidate_pairs'] for f in fold_results]
n_bp_pass = [sum(1 for p in f['pair_info'] if p['bp_pass']) for f in fold_results]
n_bp_fail = [sum(1 for p in f['pair_info'] if not p['bp_pass']) for f in fold_results]
ax.bar(fold_ids, n_bp_pass, label='BP pass (homosk.)', color='seagreen', alpha=0.8)
ax.bar(fold_ids, n_bp_fail, bottom=n_bp_pass, label='BP fail (heterosk.)', color='salmon', alpha=0.8)
ax.set_xlabel('Fold')
ax.set_ylabel('# pairs')
ax.set_title('Pairs per fold: Breusch-Pagan pass/fail')
ax.legend(fontsize=8)

# 3. BP p-value distribution
ax = axes[1, 0]
all_bp_pvals = [p['bp_pval'] for f in fold_results for p in f['pair_info']]
ax.hist(all_bp_pvals, bins=50, color='mediumpurple', alpha=0.8, edgecolor='none')
ax.axvline(0.05, color='red', ls='--', lw=1.5, label='p=0.05')
ax.set_xlabel('Breusch-Pagan p-value')
ax.set_ylabel('Count')
ax.set_title('BP p-value distribution (all pairs, all folds)')
ax.legend()

# 4. Universe size over folds
ax = axes[1, 1]
n_liquid_per_fold = [f['n_liquid'] for f in fold_results]
n_clean_per_fold = [f['n_clean'] for f in fold_results]
ax.plot(fold_ids, n_liquid_per_fold, '-o', ms=3, label='Top-150 liquid', color='C0')
ax.plot(fold_ids, n_clean_per_fold, '-o', ms=3, label='100% coverage', color='C3')
ax.set_xlabel('Fold')
ax.set_ylabel('# symbols')
ax.set_title('Universe size per fold')
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Distribution of correlations among selected pairs (those above 0.5) vs half-life
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

all_corrs_selected = []
all_hl = []
all_bp_flag = []
for f in fold_results:
    for p in f['pair_info']:
        all_corrs_selected.append(p['corr'])
        all_hl.append(p['half_life'])
        all_bp_flag.append(p['bp_pass'])

all_corrs_selected = np.array(all_corrs_selected)
all_hl = np.array(all_hl)
all_bp_flag = np.array(all_bp_flag)

# Correlation distribution of selected pairs
ax = axes[0]
ax.hist(all_corrs_selected[all_bp_flag], bins=40, alpha=0.7, label='BP pass', color='seagreen')
ax.hist(all_corrs_selected[~all_bp_flag], bins=40, alpha=0.7, label='BP fail', color='salmon')
ax.set_xlabel('Correlation')
ax.set_ylabel('Count')
ax.set_title('Correlation dist. of selected pairs')
ax.legend()

# Half-life distribution
ax = axes[1]
valid_hl = all_hl[np.isfinite(all_hl) & (all_hl > 0) & (all_hl < 500)]
valid_bp = all_bp_flag[np.isfinite(all_hl) & (all_hl > 0) & (all_hl < 500)]
ax.hist(valid_hl[valid_bp], bins=40, alpha=0.7, label='BP pass', color='seagreen')
ax.hist(valid_hl[~valid_bp], bins=40, alpha=0.7, label='BP fail', color='salmon')
ax.set_xlabel('Half-life (bars)')
ax.set_ylabel('Count')
ax.set_title('OU half-life distribution')
ax.legend()

# Scatter: correlation vs half-life
ax = axes[2]
mask_valid = np.isfinite(all_hl) & (all_hl > 0) & (all_hl < 500)
ax.scatter(all_corrs_selected[mask_valid & all_bp_flag], all_hl[mask_valid & all_bp_flag],
           s=3, alpha=0.3, label='BP pass', color='seagreen')
ax.scatter(all_corrs_selected[mask_valid & ~all_bp_flag], all_hl[mask_valid & ~all_bp_flag],
           s=3, alpha=0.3, label='BP fail', color='salmon')
ax.set_xlabel('Correlation')
ax.set_ylabel('Half-life (bars)')
ax.set_title('Correlation vs Half-life')
ax.legend()

fig.tight_layout()
plt.show()

### Walk-forward backtest with BP-filtered pairs

Using Daniel's signal machinery: OLS hedge ratio, frozen z-score, OU-based position state machine. We apply the same half-life (10–150 bars) and spread-vol filters, then trade on the test window.

In [ ]:
Z_ENTRY = 2.5
Z_EXIT = 0.5
Z_STOP = 4.0
COST_PER_SIDE = 0.0005  # 5 bps
HL_MIN, HL_MAX = 10.0, 150.0
MIN_SPREAD_VOL = 0.005
MAX_PAIRS = 20
USE_ROLLING_Z = True

all_oos_ret = []
all_oos_turn = []
all_trades = []
fold_metrics = []

for fi, fr in enumerate(fold_results):
    fold = folds[fi]
    te_close = slice_window(close_panel, fold['test_start'], fold['test_end']).ffill(limit=3)
    te_logret = slice_window(logret_panel, fold['test_start'], fold['test_end'])
    test_idx = te_close.index

    # Select tradeable pairs: BP pass + valid params + half-life/spread-vol filters
    tradeable = []
    for p in fr['pair_info']:
        if not p['bp_pass'] or not p['has_params']:
            continue
        params = p['params']
        if params.beta <= 0:
            continue
        hl = params.half_life
        if not np.isfinite(hl) or hl < HL_MIN or hl > HL_MAX:
            continue
        if not np.isfinite(params.sigma) or params.sigma < MIN_SPREAD_VOL:
            continue
        tradeable.append(params)

    # Rank by half-life (tighter = better), cap at MAX_PAIRS
    tradeable.sort(key=lambda p: p.half_life)
    tradeable = tradeable[:MAX_PAIRS]

    fold_ret = pd.Series(0.0, index=test_idx)
    fold_turn = pd.Series(0.0, index=test_idx)
    n_pairs = len(tradeable)

    if n_pairs > 0:
        wgt = 1.0 / n_pairs
        for params in tradeable:
            pr = sig.pair_returns(
                te_close, te_logret, params,
                z_entry=Z_ENTRY, z_exit=Z_EXIT, z_stop=Z_STOP,
                cost_per_side=COST_PER_SIDE, use_rolling_z=USE_ROLLING_Z,
            )
            fold_ret += pr['ret'].reindex(test_idx, fill_value=0.0) * wgt
            fold_turn += pr['turnover'].reindex(test_idx, fill_value=0.0) * wgt
            for t in pr['trades']:
                t = dict(t)
                t['fold_id'] = fi
                all_trades.append(t)

    all_oos_ret.append(fold_ret)
    all_oos_turn.append(fold_turn)

    fm = mx.summarize(fold_ret, turnover=fold_turn,
                      trade_log=pd.DataFrame([t for t in all_trades if t['fold_id'] == fi]) if any(t['fold_id'] == fi for t in all_trades) else None,
                      n_active_pairs=n_pairs)
    fm['fold_id'] = fi
    fm['test_start'] = fold['test_start'].date()
    fm['test_end'] = fold['test_end'].date()
    fm['n_tradeable'] = n_pairs
    fold_metrics.append(fm)

    if fi % 5 == 0:
        print(f"Fold {fi:2d}: {n_pairs} pairs traded, Sharpe={fm['sharpe_annualized']:+.2f}, "
              f"ret={fm['return_total']:+.3%}")

# Stitch OOS curve
oos_ret = pd.concat(all_oos_ret).sort_index()
oos_ret = oos_ret[~oos_ret.index.duplicated(keep='first')]
oos_turn = pd.concat(all_oos_turn).sort_index()
oos_turn = oos_turn[~oos_turn.index.duplicated(keep='first')]
equity = mx.equity_curve(oos_ret)

trade_log = pd.DataFrame(all_trades)
fm_df = pd.DataFrame(fold_metrics)

agg = mx.summarize(oos_ret, turnover=oos_turn,
                   trade_log=trade_log if len(trade_log) else None,
                   n_active_pairs=fm_df['n_tradeable'].mean())

print(f"\n=== AGGREGATE OOS (BP-filtered baseline) ===")
for k in ['sharpe_annualized', 'return_total', 'return_annualized',
          'max_drawdown', 'n_trades', 'trade_win_rate', 'avg_holding_bars',
          'turnover_annualized', 'n_active_pairs']:
    if k in agg:
        print(f"  {k:24s} {agg[k]:,.4f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [3, 1, 1]})

# Equity curve
ax = axes[0]
ax.plot(equity.index, equity.values, color='C0', lw=1.2)
ax.axhline(1.0, color='k', ls='--', lw=0.6)
ax.set_ylabel('Equity')
ax.set_title(f'Classic Baseline (BP-filtered, 100% cov) — '
             f'Sharpe={agg["sharpe_annualized"]:.2f}, Return={agg["return_total"]:.1%}')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# Drawdown
ax = axes[1]
dd = equity / equity.cummax() - 1.0
ax.fill_between(dd.index, dd.values, 0, color='C3', alpha=0.5)
ax.set_ylabel('Drawdown')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# Per-fold Sharpe
ax = axes[2]
test_starts = pd.to_datetime(fm_df['test_start'])
ax.bar(test_starts, fm_df['sharpe_annualized'], width=60, color='steelblue', alpha=0.8)
ax.axhline(0, color='k', lw=0.6, ls=':')
ax.set_ylabel('Fold Sharpe')
ax.set_xlabel('Fold test start')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

fig.tight_layout()
plt.show()

In [ ]:
# Trade-level analysis
if len(trade_log) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.hist(trade_log['holding_bars'], bins=40, range=(0, 200),
            color='steelblue', alpha=0.8, edgecolor='none')
    ax.set_xlabel('Holding period (bars)')
    ax.set_ylabel('# trades')
    ax.set_title(f'Holding period dist. (n={len(trade_log)})')

    ax = axes[1]
    ax.hist(trade_log['pnl'] * 100, bins=60, range=(-3, 3),
            color='steelblue', alpha=0.8, edgecolor='none')
    ax.axvline(0, color='k', ls='--', lw=0.6)
    ax.set_xlabel('Per-trade PnL (%)')
    ax.set_ylabel('# trades')
    ax.set_title(f'PnL dist. (win rate={agg.get("trade_win_rate", 0):.1%})')

    fig.tight_layout()
    plt.show()
else:
    print('No trades to plot.')

---
# Part 2: Linear Autoregression

Per stock in the top-150 liquid universe per training window, fit:

$$r_t = \sum_{i=1}^{15} \phi_i \, r_{t-i}$$

For each stock, determine the effective lookback: going backwards from lag 15, find the first lag whose coefficient < 0.01 and stop there. Average across stocks per fold to get a per-window heuristic lookback for future RNN models.

In [ ]:
MAX_LAGS = 15
COEFF_CUTOFF = 0.01

def fit_ar_and_find_lookback(returns, max_lags=MAX_LAGS, cutoff=COEFF_CUTOFF):
    """Fit AR(max_lags) on a return series (no intercept).
    Returns (coefficients, effective_lookback).
    effective_lookback: going backwards, stop at first |coeff| < cutoff."""
    r = returns.dropna()
    if len(r) < max_lags + 30:
        return None, None

    # Build lag matrix
    y = r.iloc[max_lags:].to_numpy()
    X = np.column_stack([r.shift(i).iloc[max_lags:].to_numpy() for i in range(1, max_lags + 1)])

    # Drop any remaining NaN rows
    valid = np.all(np.isfinite(X), axis=1) & np.isfinite(y)
    if valid.sum() < 30:
        return None, None
    y, X = y[valid], X[valid]

    # OLS (no intercept per the spec)
    try:
        coefs = np.linalg.lstsq(X, y, rcond=None)[0]
    except Exception:
        return None, None

    # Effective lookback: going backwards from lag 15,
    # stop at first lag with |coeff| < cutoff
    effective = max_lags
    for lag in range(max_lags, 0, -1):
        if abs(coefs[lag - 1]) < cutoff:
            effective = lag - 1
        else:
            break
    effective = max(effective, 1)  # at least 1

    return coefs, effective


# Run across all folds
ar_results = []

for fi, fold in enumerate(folds):
    tr_logret = slice_window(logret_panel, fold['train_start'], fold['train_end'])

    # Use the same liquid universe from Part 1
    liquid = get_liquid_universe_for_window(fold['train_start'], fold['train_end'])
    liquid = [s for s in liquid if s in tr_logret.columns]

    lookbacks = []
    all_coefs = []
    for sym in liquid:
        coefs, lb = fit_ar_and_find_lookback(tr_logret[sym])
        if coefs is not None:
            lookbacks.append(lb)
            all_coefs.append(coefs)

    avg_lookback = np.mean(lookbacks) if lookbacks else np.nan
    median_lookback = np.median(lookbacks) if lookbacks else np.nan
    avg_coefs = np.mean(all_coefs, axis=0) if all_coefs else np.full(MAX_LAGS, np.nan)

    ar_results.append({
        'fold_idx': fi,
        'train_start': fold['train_start'].date(),
        'train_end': fold['train_end'].date(),
        'n_stocks': len(lookbacks),
        'avg_lookback': avg_lookback,
        'median_lookback': median_lookback,
        'lookbacks': lookbacks,
        'avg_coefs': avg_coefs,
    })

    if fi % 5 == 0:
        print(f"Fold {fi:2d}: {len(lookbacks)} stocks, "
              f"avg lookback={avg_lookback:.1f}, median={median_lookback:.1f}")

ar_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'lookbacks' and k != 'avg_coefs'}
                       for r in ar_results])
print(f"\nOverall avg lookback: {ar_df['avg_lookback'].mean():.1f} lags")
print(f"Overall median lookback: {ar_df['median_lookback'].mean():.1f} lags")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Average lookback per fold over time
ax = axes[0, 0]
ax.plot(ar_df['fold_idx'], ar_df['avg_lookback'], '-o', ms=4, color='C0', label='Mean')
ax.plot(ar_df['fold_idx'], ar_df['median_lookback'], '-s', ms=4, color='C3', label='Median')
ax.set_xlabel('Fold')
ax.set_ylabel('Effective lookback (lags)')
ax.set_title('Effective AR lookback per fold')
ax.legend()

# 2. Distribution of per-stock lookbacks (pooled across folds)
ax = axes[0, 1]
all_lookbacks_pooled = [lb for r in ar_results for lb in r['lookbacks']]
ax.hist(all_lookbacks_pooled, bins=range(1, MAX_LAGS + 2), color='steelblue',
        alpha=0.8, edgecolor='white', align='left')
ax.axvline(np.mean(all_lookbacks_pooled), color='red', ls='--', lw=1.5,
           label=f'Mean={np.mean(all_lookbacks_pooled):.1f}')
ax.set_xlabel('Effective lookback (lags)')
ax.set_ylabel('Count')
ax.set_title('Distribution of per-stock effective lookback (all folds)')
ax.set_xticks(range(1, MAX_LAGS + 1))
ax.legend()

# 3. Average AR coefficients across all folds
ax = axes[1, 0]
avg_coefs_all = np.mean([r['avg_coefs'] for r in ar_results], axis=0)
colors = ['steelblue' if abs(c) >= COEFF_CUTOFF else 'lightgray' for c in avg_coefs_all]
ax.bar(range(1, MAX_LAGS + 1), avg_coefs_all, color=colors, edgecolor='white')
ax.axhline(COEFF_CUTOFF, color='red', ls='--', lw=1, label=f'|coeff|={COEFF_CUTOFF} cutoff')
ax.axhline(-COEFF_CUTOFF, color='red', ls='--', lw=1)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Lag')
ax.set_ylabel('Average coefficient')
ax.set_title('Average AR(15) coefficients (cross-fold, cross-stock)')
ax.set_xticks(range(1, MAX_LAGS + 1))
ax.legend()

# 4. Heatmap: average coefficients per fold
ax = axes[1, 1]
coef_matrix = np.array([r['avg_coefs'] for r in ar_results])
im = ax.imshow(coef_matrix.T, aspect='auto', cmap='RdBu_r',
               vmin=-0.05, vmax=0.05, origin='lower')
ax.set_xlabel('Fold')
ax.set_ylabel('Lag')
ax.set_yticks(range(MAX_LAGS))
ax.set_yticklabels(range(1, MAX_LAGS + 1))
ax.set_title('AR coefficients per fold (heatmap)')
plt.colorbar(im, ax=ax, label='Coefficient')

fig.tight_layout()
plt.show()

In [ ]:
# Summary table
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"\nPairs baseline (BP-filtered, 100% coverage):")
print(f"  Sharpe:          {agg['sharpe_annualized']:+.3f}")
print(f"  Total return:    {agg['return_total']:+.2%}")
print(f"  Max drawdown:    {agg['max_drawdown']:.2%}")
print(f"  Trades:          {agg.get('n_trades', 0):.0f}")
print(f"  Win rate:        {agg.get('trade_win_rate', 0):.1%}")
print(f"  Avg hold (bars): {agg.get('avg_holding_bars', 0):.0f}")
print(f"  Avg pairs/fold:  {agg.get('n_active_pairs', 0):.1f}")
print(f"\nAR lookback heuristic:")
print(f"  Overall avg:     {ar_df['avg_lookback'].mean():.1f} lags")
print(f"  Overall median:  {ar_df['median_lookback'].mean():.1f} lags")
print(f"  Suggested RNN lookback: {int(round(np.mean(all_lookbacks_pooled)))} bars")